# Google Colab で TEI データを分析する

このノートブックでは、架空の明治期日記『**槻村清一郎日記**』（明治20〜22年 / 1887〜1889）の TEI/XML データを、Python で読み解いて集計・可視化します。

- この日記コーパスは **教材用に生成した架空のデータ** です。登場する人物・出来事はすべて架空のもの（地名は実在）で、ライセンスは CC0 1.0 です。
- **コードは読めなくて大丈夫です。上から順に ▶ を押してください。** セルの左側にある ▶ ボタンをクリックする（または `Shift + Enter`）と、そのセルが実行されて下に結果が出ます。

## 1. データを取得する

Google Drive で公開している 3 年分の TEI ファイル（`diary-1887.xml` / `diary-1888.xml` / `diary-1889.xml`）を、この Colab 環境にダウンロードします。

In [ ]:
import urllib.request

base = "https://raw.githubusercontent.com/nakamura196/tei-analysis-tutorial/main/data/"
for year in [1887, 1888, 1889]:
    fn = f"diary-{year}.xml"
    urllib.request.urlretrieve(base + fn, fn)
    size = len(open(fn, encoding="utf-8").read())
    print(f"{fn} を取得しました({size:,} 文字)")


## 2. TEI を読み解いて表にする

TEI/XML の中から「エントリ（1日分の記事）」「人名 `<persName>`」「地名 `<placeName>`」「日付 `<date>`」を取り出して、pandas の表（DataFrame）に変換します。ここから先の分析はすべてこの表がもとになります。

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
years = [1887, 1888, 1889]

trees = {}
entry_rows, pers_rows, place_rows = [], [], []

for year in years:
    tree = ET.parse(f"diary-{year}.xml")
    trees[year] = tree
    body = tree.getroot().find("tei:text/tei:body", NS)
    for div in body.findall("tei:div[@type='entry']", NS):
        when = div.find("tei:head/tei:date", NS).get("when")
        text = "".join(div.find("tei:p", NS).itertext())
        entry_rows.append({"年": year, "日付": when, "月": int(when[5:7]), "本文": text})
        for el in div.findall(".//tei:persName", NS):
            pers_rows.append({"年": year, "日付": when, "人名": el.text})
        for el in div.findall(".//tei:placeName", NS):
            place_rows.append({"年": year, "日付": when, "地名": el.text})

entries_df = pd.DataFrame(entry_rows)
pers_df = pd.DataFrame(pers_rows)
place_df = pd.DataFrame(place_rows)

print(f"エントリ: {len(entries_df)} 件 / 人名: {len(pers_df)} 件 / 地名: {len(place_df)} 件")
print(entries_df["年"].value_counts().sort_index().rename("年別エントリ数"))
entries_df.head()

## 3. よく登場する人物は？

人名の出現回数を数えて、上位 10 名を棒グラフにします。グラフの日本語が文字化けしないように、最初に日本語フォント（japanize-matplotlib）を入れます（初回のみ数秒かかります）。

In [ ]:
!pip install japanize-matplotlib -q
import japanize_matplotlib
import matplotlib.pyplot as plt

top10 = pers_df["人名"].value_counts().head(10)

ax = top10.plot(kind="bar", figsize=(10, 5), color="steelblue")
ax.set_title("よく登場する人物 上位10名（1887〜1889年）")
ax.set_ylabel("登場回数")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. 日記はいつ書かれたか

月別のエントリ数を、3 年分重ねて折れ線グラフにします。年ごとの書きぶりの波が見えてきます。

In [ ]:
monthly = entries_df.groupby(["月", "年"]).size().unstack("年")

ax = monthly.plot(figsize=(10, 5), marker="o")
ax.set_title("月別エントリ数（3年分の重ね書き）")
ax.set_xlabel("月")
ax.set_ylabel("エントリ数")
ax.set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 5. 地名を地図に置いてみる

地名を地図に置くには緯度・経度が必要ですが、**その台帳は TEI ファイル自身の中に入っています**。ファイル冒頭近くの `<standOff>` の中に `<listPlace>`（地名の一覧）があり、各地名の座標が `<geo>` 要素に記録されています。

```xml
<place xml:id="hongo">
  <placeName>本郷</placeName>
  <location><geo>35.7081 139.7620</geo></location>
</place>
```

本文中の `<placeName ref="#hongo">本郷</placeName>` がこの台帳を参照しています。外部の対応表を別に用意しなくても、TEI ファイルだけで地図が描ける、というわけです。円の大きさが出現回数に対応し、円をクリックすると地名と回数が表示されます。

In [ ]:
import math
import folium

root = trees[1887].getroot()
gaz_rows = []
for place in root.findall("tei:standOff/tei:listPlace/tei:place", NS):
    name = place.find("tei:placeName", NS).text
    lat, lon = map(float, place.find("tei:location/tei:geo", NS).text.split())
    gaz_rows.append({"地名": name, "緯度": lat, "経度": lon})

gazetteer_df = pd.DataFrame(gaz_rows)
print(f"TEI の台帳（standOff/listPlace）に載っている地名: {len(gazetteer_df)} 箇所")

counts = place_df["地名"].value_counts().rename("出現回数")
map_df = gazetteer_df.merge(counts, left_on="地名", right_index=True, how="left")
map_df["出現回数"] = map_df["出現回数"].fillna(0).astype(int)
display(map_df.sort_values("出現回数", ascending=False).head(10))

m = folium.Map(location=[35.5, 136.5], zoom_start=6)
for _, row in map_df.iterrows():
    if row["出現回数"] == 0:
        continue
    folium.CircleMarker(
        location=[row["緯度"], row["経度"]],
        radius=max(4, math.sqrt(row["出現回数"]) * 2.5),
        color="crimson",
        fill=True,
        fill_opacity=0.6,
        popup=f"{row['地名']}（{row['出現回数']} 回）",
    ).add_to(m)
m

## 6. Gemini に頼んでみよう（その1）

Colab には Gemini（Google の AI）が組み込まれています。コードセルの右上に出る「✨」アイコンや「コードを生成」から、日本語で依頼できます。

下の依頼文をコピーして Gemini に頼み、生成されたコードを次の空のセルに入れて ▶ で実行してみてください。

```text
人名の集計（上のセルの棒グラフ）を、地名の集計に変えてください。
place_df の「地名」列を使い、上位10地名の棒グラフを表示してください。
```

## 6. Gemini に頼んでみよう（その2）

今度は「集計」ではなく「本文の抜き出し」を頼んでみます。1888（明治21）年の夏、書き手は京都・大阪・奈良へ旅行しています。

```text
1888年7月と8月の旅行（京都・大阪・奈良）の日記本文だけを抜き出して表示してください。
entries_df の「日付」列（YYYY-MM-DD 形式）と「本文」列が使えます。
本文に「京都」「大阪」「奈良」のいずれかを含むエントリを、日付つきで表示してください。
```

## 7. 結果をファイルに書き出す

最後に、分析結果を手元のパソコンにダウンロードします。

- 年ごとのプレーンテキスト（`diary-1887.txt` など3つ）— **次のパート（Voyant Tools）でそのまま使えます**
- 人名の集計 CSV（`persons.csv`）— **RAWGraphs でそのまま使えます**

ブラウザが「複数ファイルのダウンロードを許可しますか」と聞いてきたら「許可」を選んでください。

In [ ]:
from google.colab import files

for year in years:
    body = trees[year].getroot().find("tei:text/tei:body", NS)
    lines = []
    for div in body.findall("tei:div[@type='entry']", NS):
        lines.append("".join(div.find("tei:head", NS).itertext()))
        lines.append("".join(div.find("tei:p", NS).itertext()))
        lines.append("")
    Path(f"diary-{year}.txt").write_text("\n".join(lines), encoding="utf-8")
    print(f"diary-{year}.txt を書き出しました")

pers_counts = pers_df["人名"].value_counts().rename_axis("人名").reset_index(name="回数")
pers_counts.to_csv("persons.csv", index=False, encoding="utf-8-sig")
print("persons.csv を書き出しました")

for fname in [f"diary-{year}.txt" for year in years] + ["persons.csv"]:
    files.download(fname)

## おつかれさまでした

ここまでで、TEI データの取得 → 表への変換 → 集計・グラフ → 地図 → 書き出し、という分析の一連の流れを体験しました。

もっと先へ進みたい方は、配布フォルダにある発展教材の PDF『**Google Colab で分析する TEI データ — Python で頻度・共起・時系列を扱う実践ガイド**』をご覧ください。このノートブックと同じコーパスを使って、共起分析・時系列の深掘り・3年分の文書比較などを、Gemini への依頼のしかたとあわせて解説しています。